## Aurora worker mode (LatentSync + MimicMotion)
The demo pipeline above is GPU-free and great for testing. **Section 13** turns
this notebook into a real [Aurora](./) self-hosted GPU worker: it runs
**LatentSync** (lip-sync) and **MimicMotion** (motion transfer) and serves them
over Aurora's flat `/generate` job contract (see `workers/CONTRACT.md`).

With a free **Ngrok static domain** + the `AURORA_*` secrets (see the Section 13
setup cell), the worker keeps the **same URL across restarts** and **registers
itself** in `gpu_workers` on boot — so you never have to touch **Admin → Workers**
again. Capabilities are `lipsync,motion`.


## 1 · Setup & configuration

In [ ]:
#@title Install dependencies (quiet) { display-mode: "form" }
# Minimal, free-tier friendly installs. torch already ships with Colab.
import sys, subprocess

PKGS = [
    "diffusers>=0.27",
    "transformers>=4.40",
    "accelerate>=0.30",
    "safetensors",
    "gradio==4.44.0",
    "imageio",
    "imageio-ffmpeg",
    "soundfile",
    "librosa",
    "requests",
]

def _pip_install(pkgs):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs],
        check=False,
    )

print("Installing packages (cached on re-runs)...")
_pip_install(PKGS)
print("Done.")

In [ ]:
#@title Configuration
import os

CONFIG = {
    # ---- Models (lightweight, free-tier friendly) ----
    "TXT2IMG_MODEL":    "stabilityai/sd-turbo",     # fast 1-4 step model (low VRAM)
    "TXT2IMG_MODEL_HQ": "stabilityai/sdxl-turbo",   # used only with lots of VRAM
    "ENABLE_SVD": False,   # Stable Video Diffusion (heavy ~10GB) -- keep off on free tier

    # ---- Compute / fallback ----
    "COMPUTE_MODE": "auto",   # "auto" | "local" | "remote"
    "LOW_VRAM": True,         # attention/vae slicing + CPU offload
    "MAX_LOCAL_RETRIES": 2,
    "REMOTE_RETRIES": 3,

    # ---- External GPU API (RunPod-style). Leave blank to disable remote fallback ----
    "RUNPOD_API_KEY":     os.environ.get("RUNPOD_API_KEY", ""),
    "RUNPOD_ENDPOINT_ID": os.environ.get("RUNPOD_ENDPOINT_ID", ""),
    # Optional generic REST endpoint ("or similar"): must accept {"prompt": ...}
    # and return JSON containing an image URL or base64. Leave blank to ignore.
    "REMOTE_API_URL": os.environ.get("REMOTE_API_URL", ""),
    "REMOTE_API_KEY": os.environ.get("REMOTE_API_KEY", ""),

    # ---- Persistence ----
    "USE_DRIVE": True,
    "PROJECT_DIR_NAME": "hybrid_ai_engine",
}

print("Configuration loaded. Edit the CONFIG dict above to customise.")

## 2 · Environment & hardware detection
Detects whether we are on Colab / Kaggle / local, and what accelerator is available.

In [ ]:
#@title Detect environment + hardware
import os, multiprocessing, torch

def detect_environment():
    env = "local"
    if "COLAB_GPU" in os.environ or os.path.isdir("/content"):
        env = "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle"):
        env = "kaggle"
    return env

def detect_hardware():
    info = {
        "env": detect_environment(),
        "has_gpu": torch.cuda.is_available(),
        "gpu_name": None,
        "vram_gb": 0.0,
        "has_tpu": ("COLAB_TPU_ADDR" in os.environ) or ("TPU_NAME" in os.environ),
        "cpu_count": multiprocessing.cpu_count(),
        "device": "cpu",
    }
    if info["has_gpu"]:
        info["device"] = "cuda"
        props = torch.cuda.get_device_properties(0)
        info["gpu_name"] = props.name
        info["vram_gb"] = round(props.total_memory / (1024 ** 3), 1)
    return info

HW = detect_hardware()
DEVICE = HW["device"]
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Environment :", HW["env"])
print("Device      :", DEVICE)
print("GPU         :", HW["gpu_name"], (("(%.1f GB)" % HW["vram_gb"]) if HW["has_gpu"] else ""))
print("TPU present :", HW["has_tpu"])
print("CPU cores   :", HW["cpu_count"])
if not HW["has_gpu"]:
    print("\n[!] No GPU detected. Enable one via Runtime -> Change runtime type, or rely on the remote API fallback.")

## 3 · Persistence — storage mount + model cache
Mounts Google Drive (Colab) so outputs **and** downloaded model weights survive
across sessions. Falls back to Kaggle working dir or a local folder.

In [ ]:
#@title Mount storage + set model cache
import os

def setup_storage(cfg, hw):
    base = None
    # Colab -> Google Drive (persists across sessions)
    if cfg["USE_DRIVE"] and hw["env"] == "colab":
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            base = "/content/drive/MyDrive/" + cfg["PROJECT_DIR_NAME"]
        except Exception as e:
            print("Drive mount failed (%s); using local storage." % e)
    # Kaggle -> working dir
    if base is None and hw["env"] == "kaggle":
        base = "/kaggle/working/" + cfg["PROJECT_DIR_NAME"]
    # Anything else -> local folder
    if base is None:
        base = os.path.abspath("./" + cfg["PROJECT_DIR_NAME"])

    outputs = os.path.join(base, "outputs")
    cache = os.path.join(base, "hf_cache")
    os.makedirs(outputs, exist_ok=True)
    os.makedirs(cache, exist_ok=True)

    # Persist HF model weights so they are not re-downloaded next session.
    os.environ["HF_HOME"] = cache
    os.environ["HUGGINGFACE_HUB_CACHE"] = cache
    return base, outputs

BASE_DIR, OUTPUT_DIR = setup_storage(CONFIG, HW)
print("Base dir :", BASE_DIR)
print("Outputs  :", OUTPUT_DIR)
print("HF cache :", os.environ["HF_HOME"])

## 4 · Reconnect / keep-alive helpers
Stops Colab from dropping an **idle** tab, and provides resumable state on Drive.
If Google fully recycles the runtime, just re-run the notebook — the model cache
and outputs are restored from Drive.

In [ ]:
#@title Anti-idle keep-alive + resumable state
import json, os
from IPython.display import display, Javascript

def enable_keep_alive():
    try:
        display(Javascript("""
            (function(){
              if (window.__keepAlive) return;
              window.__keepAlive = setInterval(function(){
                try {
                  var b = document.querySelector("colab-toolbar-button#connect");
                  if (b) { b.click(); }
                } catch (e) {}
              }, 60000);
              console.log("keep-alive enabled");
            })();
        """))
        print("Keep-alive enabled.")
    except Exception as e:
        print("Keep-alive not available here:", e)

_STATE_PATH = os.path.join(BASE_DIR, "state.json")

def save_state(state):
    try:
        with open(_STATE_PATH, "w") as f:
            json.dump(state, f, indent=2)
    except Exception as e:
        print("Could not save state:", e)

def load_state():
    try:
        with open(_STATE_PATH) as f:
            return json.load(f)
    except Exception:
        return {}

enable_keep_alive()

## 5 · Memory management + low-VRAM mode

In [ ]:
#@title Memory helpers
import gc, torch

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def apply_optimizations(pipe, low_vram=True):
    # Diffusers pipeline tweaks; each is a safe no-op if unavailable.
    for name in ["enable_attention_slicing", "enable_vae_slicing", "enable_vae_tiling"]:
        fn = getattr(pipe, name, None)
        if callable(fn):
            try:
                fn()
            except Exception:
                pass
    if low_vram:
        try:
            pipe.enable_model_cpu_offload()
        except Exception:
            pipe.to(DEVICE)
    else:
        pipe.to(DEVICE)
    return pipe

## 6 · Model loading (lazy + cached)
Models are loaded **on first use** and kept in memory. SDXL-Turbo is only chosen
when there is comfortable VRAM headroom; otherwise the lighter SD-Turbo is used.

In [ ]:
#@title Lazy model registry
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image

_PIPES = {}

def _choose_txt2img_model():
    if HW["has_gpu"] and HW["vram_gb"] >= 15 and not CONFIG["LOW_VRAM"]:
        return CONFIG["TXT2IMG_MODEL_HQ"]
    return CONFIG["TXT2IMG_MODEL"]

def get_txt2img():
    if "txt2img" not in _PIPES:
        model = _choose_txt2img_model()
        print("Loading text-to-image model:", model)
        pipe = AutoPipelineForText2Image.from_pretrained(model, torch_dtype=DTYPE)
        _PIPES["txt2img"] = apply_optimizations(pipe, CONFIG["LOW_VRAM"])
        _PIPES["txt2img_model"] = model
    return _PIPES["txt2img"]

def get_img2img():
    if "img2img" not in _PIPES:
        model = _PIPES.get("txt2img_model") or _choose_txt2img_model()
        print("Loading image-to-image model:", model)
        if "txt2img" in _PIPES:
            # Reuse weights already in memory.
            pipe = AutoPipelineForImage2Image.from_pipe(_PIPES["txt2img"])
        else:
            pipe = AutoPipelineForImage2Image.from_pretrained(model, torch_dtype=DTYPE)
            pipe = apply_optimizations(pipe, CONFIG["LOW_VRAM"])
            _PIPES["txt2img_model"] = model
        _PIPES["img2img"] = pipe
    return _PIPES["img2img"]

def unload_all():
    for k in list(_PIPES.keys()):
        obj = _PIPES.pop(k, None)
        del obj
    free_memory()
    print("Unloaded all models.")

## 7 · Pipeline stages (image / video / audio-driven)

In [ ]:
#@title Stage: text -> image  /  image -> image
import torch
from PIL import Image

def _turbo_kwargs(model_id):
    # *-turbo models use very few steps and no classifier-free guidance.
    if "turbo" in (model_id or ""):
        return dict(num_inference_steps=2, guidance_scale=0.0)
    return dict(num_inference_steps=25, guidance_scale=7.0)

def generate_image(prompt, init_image=None, strength=0.6, seed=None, width=512, height=512):
    generator = None
    if seed is not None:
        generator = torch.Generator(device=DEVICE).manual_seed(int(seed))

    if init_image is None:
        pipe = get_txt2img()
        kw = _turbo_kwargs(_PIPES.get("txt2img_model"))
        out = pipe(prompt=prompt, width=width, height=height, generator=generator, **kw)
    else:
        pipe = get_img2img()
        kw = _turbo_kwargs(_PIPES.get("txt2img_model"))
        init_image = init_image.convert("RGB").resize((width, height))
        out = pipe(prompt=prompt, image=init_image, strength=strength, generator=generator, **kw)
    return out.images[0]

In [ ]:
#@title Stage: image -> short video (lightweight Ken Burns, GPU-free)
import numpy as np, imageio
from PIL import Image

def _block_resize(img, mult=16, max_side=768):
    img = img.convert("RGB")
    W, H = img.size
    scale = min(1.0, max_side / float(max(W, H)))
    W, H = int(W * scale), int(H * scale)
    W = max(mult, W - W % mult)
    H = max(mult, H - H % mult)
    return img.resize((W, H), Image.LANCZOS)

def ken_burns_video(image, out_path, seconds=3, fps=16, zoom=1.25):
    img = _block_resize(image)
    W, H = img.size
    n = max(2, int(seconds * fps))
    frames = []
    for i in range(n):
        t = i / (n - 1)
        z = 1.0 + (zoom - 1.0) * t
        cw, ch = int(W / z), int(H / z)
        x = int((W - cw) * t)
        y = int((H - ch) * t)
        crop = img.crop((x, y, x + cw, y + ch)).resize((W, H), Image.LANCZOS)
        frames.append(np.array(crop))
    imageio.mimwrite(out_path, frames, fps=fps, codec="libx264", quality=8)
    return out_path

In [ ]:
#@title Stage: audio-driven animation (lightweight, GPU-free)
# NOTE: this is amplitude-driven motion, not true phoneme lip-sync. See the final
# notes cell for how to plug in Wav2Lip / SadTalker for real lip-sync.
import os, subprocess, numpy as np, imageio
from PIL import Image

def _ffmpeg_exe():
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return "ffmpeg"

def envelope_from_signal(y, sr, fps, n_frames):
    if y is None or len(y) == 0:
        t = np.linspace(0, 6.2832, n_frames)
        return np.sin(t * 3.0) * 0.5 + 0.5
    try:
        import librosa
        hop = max(1, int(sr / fps))
        rms = librosa.feature.rms(y=y, frame_length=hop * 2, hop_length=hop)[0]
    except Exception:
        win = max(1, int(sr / fps))
        nb = max(1, len(y) // win)
        rms = np.array([np.sqrt(np.mean(y[i*win:(i+1)*win] ** 2) + 1e-9) for i in range(nb)])
    if rms.max() > 0:
        rms = rms / rms.max()
    idx = np.linspace(0, len(rms) - 1, n_frames).astype(int)
    return rms[idx]

def audio_driven_video(image, audio_path, out_path, fps=16, max_seconds=12):
    try:
        import librosa
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        dur = len(y) / float(sr) if sr else 4.0
    except Exception as e:
        print("Could not read audio (%s); using idle motion." % e)
        y, sr, dur = None, 16000, 4.0

    dur = float(min(max_seconds, max(1.0, dur)))
    n = max(2, int(dur * fps))
    env = envelope_from_signal(y, sr, fps, n)

    base = _block_resize(image)
    W, H = base.size
    frames = []
    for i in range(n):
        a = float(env[i])
        z = 1.0 + 0.04 * a              # subtle "speaking" pulse
        cw, ch = int(W / z), int(H / z)
        x = (W - cw) // 2
        y0 = (H - ch) // 2
        crop = base.crop((x, y0, x + cw, y0 + ch)).resize((W, H), Image.LANCZOS)
        frames.append(np.array(crop))

    silent = out_path.replace(".mp4", "_silent.mp4")
    imageio.mimwrite(silent, frames, fps=fps, codec="libx264", quality=8)

    try:
        cmd = [_ffmpeg_exe(), "-y", "-i", silent, "-i", audio_path,
               "-c:v", "copy", "-c:a", "aac", "-shortest", out_path]
        subprocess.run(cmd, check=True, capture_output=True)
        if os.path.exists(out_path):
            os.remove(silent)
        else:
            out_path = silent
    except Exception as e:
        print("Audio mux failed (%s); returning silent video." % e)
        out_path = silent
    return out_path

## 8 · External GPU API fallback (RunPod-style REST)
Sends the job to a serverless GPU endpoint when the local GPU is unavailable or
exhausted. Includes retries with exponential backoff and flexible output parsing.

In [ ]:
#@title RunPod-style remote client (REST + retries)
import time, base64, io, requests
from PIL import Image

class RemoteError(Exception):
    pass

def _b64_or_url_to_image(value):
    if isinstance(value, str) and value.startswith("http"):
        r = requests.get(value, timeout=120)
        r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    if isinstance(value, str) and "," in value[:64]:
        value = value.split(",", 1)[1]   # strip data-URI prefix
    return Image.open(io.BytesIO(base64.b64decode(value))).convert("RGB")

def _find_image_field(obj):
    # Walk a JSON response looking for an image URL or base64 blob.
    if isinstance(obj, str) and (obj.startswith("http") or len(obj) > 256):
        return obj
    if isinstance(obj, dict):
        for k in ["image", "image_url", "url", "output", "images", "image_base64", "base64"]:
            if k in obj:
                found = _find_image_field(obj[k])
                if found:
                    return found
        for v in obj.values():
            found = _find_image_field(v)
            if found:
                return found
    if isinstance(obj, list) and obj:
        return _find_image_field(obj[0])
    return None

def runpod_generate(prompt, **kwargs):
    key = CONFIG["RUNPOD_API_KEY"]
    endpoint = CONFIG["RUNPOD_ENDPOINT_ID"]
    if not key or not endpoint:
        raise RemoteError("RunPod not configured (set RUNPOD_API_KEY + RUNPOD_ENDPOINT_ID).")

    base = "https://api.runpod.ai/v2/" + endpoint
    headers = {"Authorization": "Bearer " + key, "Content-Type": "application/json"}
    payload = {"input": {"prompt": prompt, **kwargs}}

    last = None
    for attempt in range(CONFIG["REMOTE_RETRIES"]):
        try:
            r = requests.post(base + "/run", json=payload, headers=headers, timeout=60)
            r.raise_for_status()
            job = r.json()
            job_id = job.get("id")
            status = job.get("status")
            while status in ("IN_QUEUE", "IN_PROGRESS"):
                time.sleep(2)
                s = requests.get(base + "/status/" + job_id, headers=headers, timeout=60)
                s.raise_for_status()
                job = s.json()
                status = job.get("status")
            if status == "COMPLETED":
                img = _find_image_field(job.get("output"))
                if img is None:
                    raise RemoteError("Remote job completed but no image in output.")
                return _b64_or_url_to_image(img)
            raise RemoteError("Remote job status: %s" % status)
        except Exception as e:
            last = e
            wait = 2 ** attempt
            print("Remote attempt %d failed (%s); retrying in %ds..." % (attempt + 1, e, wait))
            time.sleep(wait)
    raise RemoteError("Remote generation failed after retries: %s" % last)

def generic_remote_generate(prompt, **kwargs):
    url = CONFIG["REMOTE_API_URL"]
    if not url:
        raise RemoteError("No generic REMOTE_API_URL configured.")
    headers = {"Content-Type": "application/json"}
    if CONFIG["REMOTE_API_KEY"]:
        headers["Authorization"] = "Bearer " + CONFIG["REMOTE_API_KEY"]
    last = None
    for attempt in range(CONFIG["REMOTE_RETRIES"]):
        try:
            r = requests.post(url, json={"prompt": prompt, **kwargs}, headers=headers, timeout=180)
            r.raise_for_status()
            img = _find_image_field(r.json())
            if img is None:
                raise RemoteError("No image field in remote response.")
            return _b64_or_url_to_image(img)
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise RemoteError("Generic remote failed: %s" % last)

## 9 · Smart orchestrator (auto fallback + retries)
The "brain": tries local → low-VRAM → external API, detecting usage-limit / OOM
errors and switching compute sources automatically.

In [ ]:
#@title Compute orchestrator
import torch

LIMIT_SIGNS = ["out of memory", "cuda error", "cublas", "device-side assert",
               "usage limit", "resource exhausted", "no cuda gpus are available"]

def _looks_like_limit(exc):
    msg = str(exc).lower()
    return any(s in msg for s in LIMIT_SIGNS)

def _remote_available():
    return (bool(CONFIG["RUNPOD_API_KEY"]) and bool(CONFIG["RUNPOD_ENDPOINT_ID"])) \
        or bool(CONFIG["REMOTE_API_URL"])

def remote_image(prompt, **kwargs):
    if CONFIG["RUNPOD_API_KEY"] and CONFIG["RUNPOD_ENDPOINT_ID"]:
        return runpod_generate(prompt, **kwargs)
    return generic_remote_generate(prompt, **kwargs)

def smart_generate_image(prompt, log=print, **kwargs):
    mode = CONFIG["COMPUTE_MODE"]

    if mode == "remote":
        log("Compute mode = remote -> external GPU API.")
        return remote_image(prompt, **kwargs)

    if mode == "auto" and not HW["has_gpu"]:
        if _remote_available():
            log("No local GPU -> external GPU API.")
            return remote_image(prompt, **kwargs)
        log("No local GPU and no remote API -> running on CPU (slow).")

    retries = CONFIG["MAX_LOCAL_RETRIES"]
    for attempt in range(retries + 1):
        try:
            return generate_image(prompt, **kwargs)
        except Exception as e:
            log("Local attempt %d failed: %s" % (attempt + 1, e))
            free_memory()
            if _looks_like_limit(e):
                if not CONFIG["LOW_VRAM"]:
                    log("Enabling low-VRAM mode and retrying...")
                    CONFIG["LOW_VRAM"] = True
                    unload_all()
                    continue
                if mode == "auto" and _remote_available():
                    log("Local GPU exhausted -> switching to external GPU API.")
                    return remote_image(prompt, **kwargs)
            if attempt == retries:
                raise
    raise RuntimeError("Image generation failed on all backends.")

## 10 · Gradio UI
Upload an input, choose a pipeline, preview the result. A public share link is printed.

In [ ]:
#@title Launch the app
import os, time, traceback, gradio as gr
from PIL import Image

def _save(name_tpl, save_fn):
    ts = time.strftime("%Y%m%d_%H%M%S")
    path = os.path.join(OUTPUT_DIR, name_tpl % ts)
    save_fn(path)
    return path

def run_pipeline(task, prompt, init_image, audio, do_animate, low_vram,
                 compute_mode, duration, seed):
    logs = []
    def log(m):
        logs.append(str(m)); print(m)

    CONFIG["LOW_VRAM"] = bool(low_vram)
    CONFIG["COMPUTE_MODE"] = compute_mode
    out_image, out_video = None, None

    try:
        if task == "Text -> Image":
            img = smart_generate_image(prompt, log=log, seed=(seed if seed >= 0 else None))
            out_image = img
            p = _save("image_%s.png", lambda pth: img.save(pth))
            log("Saved image -> " + p)
            if do_animate:
                vp = _save("video_%s.mp4", lambda pth: ken_burns_video(img, pth, seconds=duration))
                out_video = vp; log("Saved animation -> " + vp)

        elif task == "Image -> Video":
            if init_image is None:
                log("Please upload an image."); return None, None, "\n".join(logs)
            img = init_image
            if prompt.strip():
                img = smart_generate_image(prompt, log=log, init_image=init_image, strength=0.5)
                out_image = img
            vp = _save("video_%s.mp4", lambda pth: ken_burns_video(img, pth, seconds=duration))
            out_video = vp; log("Saved animation -> " + vp)

        elif task == "Audio-driven animation":
            if init_image is None or audio is None:
                log("Please upload BOTH an image and an audio file.")
                return None, None, "\n".join(logs)
            vp = _save("talking_%s.mp4",
                       lambda pth: audio_driven_video(init_image, audio, pth, max_seconds=duration))
            out_video = vp; log("Saved audio-driven video -> " + vp)

        free_memory()
    except Exception as e:
        log("ERROR: " + str(e))
        log(traceback.format_exc())

    return out_image, out_video, "\n".join(logs)

with gr.Blocks(title="Hybrid AI Inference Engine") as demo:
    gr.Markdown("# 🧠 Hybrid AI Inference Engine\n"
                "Text -> Image -> Video / Audio-driven, with automatic GPU fallback.")
    with gr.Row():
        with gr.Column():
            task = gr.Dropdown(
                ["Text -> Image", "Image -> Video", "Audio-driven animation"],
                value="Text -> Image", label="Pipeline")
            prompt = gr.Textbox(label="Prompt",
                                value="a cinematic portrait of an astronaut, soft light")
            init_image = gr.Image(label="Input image (optional)", type="pil")
            audio = gr.Audio(label="Input audio (for audio-driven)", type="filepath")
            with gr.Row():
                do_animate = gr.Checkbox(label="Also animate result", value=False)
                low_vram = gr.Checkbox(label="Low-VRAM mode", value=CONFIG["LOW_VRAM"])
            compute_mode = gr.Radio(["auto", "local", "remote"],
                                    value=CONFIG["COMPUTE_MODE"], label="Compute")
            duration = gr.Slider(1, 12, value=3, step=1, label="Video seconds")
            seed = gr.Number(value=-1, label="Seed (-1 = random)", precision=0)
            run = gr.Button("Generate", variant="primary")
        with gr.Column():
            out_img = gr.Image(label="Image output")
            out_vid = gr.Video(label="Video output")
            out_log = gr.Textbox(label="Log", lines=12)

    run.click(run_pipeline,
              inputs=[task, prompt, init_image, audio, do_animate, low_vram,
                      compute_mode, duration, seed],
              outputs=[out_img, out_vid, out_log])

demo.launch(share=True, debug=False)

## 11 · (Optional) quick smoke test without the UI

In [ ]:
#@title Smoke test (optional)
try:
    img = smart_generate_image("a tiny robot watering a plant, studio light")
    test_path = os.path.join(OUTPUT_DIR, "smoke_test.png")
    img.save(test_path)
    print("OK -> saved", test_path)
    img
except Exception as e:
    print("Smoke test failed:", e)

## 13 · Aurora worker mode — LatentSync (lipsync) + MimicMotion (motion)

Runs the **real** self-hosted models and serves Aurora's flat `/generate` contract
behind a public tunnel. Needs a **GPU runtime** and a ~24 GB one-time weights
download. This mirrors `workers/aurora_worker.py`.


### 🔑 Zero-touch setup — stable URL + auto-register

Set these as **Colab secrets** (the 🔑 icon in the left sidebar) and the next cell
loads them. The worker then gets the **same URL on every restart** and **registers
itself** in Aurora — you never edit **Admin → Workers** again.

| secret | what it is |
| --- | --- |
| `NGROK_AUTHTOKEN` | your Ngrok account token (dashboard.ngrok.com → *Your Authtoken*) |
| `NGROK_STATIC_DOMAIN` | your free static domain, e.g. `foo-bar.ngrok-free.app` |
| `AURORA_URL` | base URL of your Aurora app, e.g. `https://aurora.example.com` |
| `AURORA_REGISTER_SECRET` | private operator secret you generate; set the same value in Aurora's env (sent as the `apikey` header) |
| `AURORA_WORKER_TOKEN` | *(optional)* bearer that protects this worker's `/generate` |

**Get a free static domain (one per account):** open
<https://dashboard.ngrok.com/domains> → **+ New Domain** → copy the
`*.ngrok-free.app` name into `NGROK_STATIC_DOMAIN`.

Once these are set, **Run all** after any Colab restart and the worker comes back
online automatically — no copy-pasting endpoint URLs.


In [ ]:
#@title 🔑 Load Aurora worker secrets (from the Colab 🔑 sidebar) { display-mode: "form" }
import os
_KEYS = ["NGROK_AUTHTOKEN", "NGROK_STATIC_DOMAIN", "AURORA_URL",
         "AURORA_REGISTER_SECRET", "AURORA_WORKER_TOKEN", "AURORA_WORKER_NAME"]
try:
    from google.colab import userdata
    for _k in _KEYS:
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            pass  # secret not set / access not granted — that's fine
except Exception:
    pass  # not running on Colab — set the env vars yourself

_set = [k for k in _KEYS if os.environ.get(k)]
print("Loaded:", ", ".join(_set) if _set else "none")
if not all(os.environ.get(k) for k in ("NGROK_STATIC_DOMAIN", "AURORA_URL", "AURORA_REGISTER_SECRET")):
    print("[!] For auto-register set NGROK_STATIC_DOMAIN, AURORA_URL and AURORA_REGISTER_SECRET.")


In [ ]:
#@title Aurora worker — serve LatentSync + MimicMotion over /generate { display-mode: "form" }
# Real Aurora worker: implements the flat job contract in workers/CONTRACT.md.
#   kind="lipsync" -> LatentSync   (video_url + audio_url)
#   kind="motion"  -> MimicMotion  (image_urls[0] + video_url)
#   kind="image"   -> SD/SDXL-Turbo (prompt [+ image_urls[0]]) — advertised ONLY
#                     when a GPU image model actually loads; omitted otherwise.
# Returns {"url": ...}. Register the printed URL as a `custom` worker; the caps it
# advertises are printed at startup (lipsync, motion [, image]).
import os, uuid, subprocess, tempfile, mimetypes
from pathlib import Path
import requests

ROOT = "/content"
LATENTSYNC_DIR = os.environ.setdefault("LATENTSYNC_DIR", f"{ROOT}/LatentSync")
MIMICMOTION_DIR = os.environ.setdefault("MIMICMOTION_DIR", f"{ROOT}/MimicMotion")
WORK = Path(tempfile.gettempdir()) / "aurora"; WORK.mkdir(parents=True, exist_ok=True)
UPLOAD = os.environ.get("AURORA_UPLOAD", "catbox")  # "catbox" (no account) or "supabase"

# 1) One-time setup: clone repos + fetch weights. (Re-running is a no-op once present.)
if not os.path.isdir(f"{LATENTSYNC_DIR}/checkpoints"):
    !git clone -q https://github.com/bytedance/LatentSync.git {LATENTSYNC_DIR}
    !pip -q install -r {LATENTSYNC_DIR}/requirements.txt
    !huggingface-cli download ByteDance/LatentSync-1.5 --local-dir {LATENTSYNC_DIR}/checkpoints --include "latentsync_unet.pt" "whisper/*"
if not os.path.isdir(f"{MIMICMOTION_DIR}/models"):
    !git clone -q https://github.com/Tencent/MimicMotion.git {MIMICMOTION_DIR}
    !pip -q install -r {MIMICMOTION_DIR}/requirements.txt
    !huggingface-cli download tencent/MimicMotion MimicMotion_1-1.pth --local-dir {MIMICMOTION_DIR}/models

def _dl(url, suf):
    p = WORK / f"{uuid.uuid4().hex}{suf}"
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status(); p.write_bytes(r.content)
    return str(p)

def _up(path):
    if UPLOAD == "supabase":
        base = os.environ["SUPABASE_URL"].rstrip("/"); key = os.environ["SUPABASE_SERVICE_ROLE_KEY"]
        bkt = os.environ.get("SUPABASE_BUCKET", "studio")
        name = f"worker/{uuid.uuid4().hex}{os.path.splitext(path)[1]}"
        ct = mimetypes.guess_type(path)[0] or "application/octet-stream"
        r = requests.post(f"{base}/storage/v1/object/{bkt}/{name}",
                          headers={"authorization": f"Bearer {key}", "content-type": ct, "x-upsert": "true"},
                          data=open(path, "rb").read(), timeout=300); r.raise_for_status()
        return f"{base}/storage/v1/object/public/{bkt}/{name}"
    r = requests.post("https://catbox.moe/user/api.php", data={"reqtype": "fileupload"},
                      files={"fileToUpload": open(path, "rb")}, timeout=180); r.raise_for_status()
    return r.text.strip()

def _latentsync(video_url, audio_url, p):
    v = _dl(video_url, ".mp4"); a = _dl(audio_url, ".wav"); out = str(WORK / f"{uuid.uuid4().hex}.mp4")
    subprocess.run(["python", "-m", "scripts.inference",
                    "--unet_config_path", f"{LATENTSYNC_DIR}/configs/unet/stage2.yaml",
                    "--inference_ckpt_path", f"{LATENTSYNC_DIR}/checkpoints/latentsync_unet.pt",
                    "--inference_steps", str(int(p.get("inference_steps", 20))),
                    "--guidance_scale", str(float(p.get("guidance_scale", 1.5))),
                    "--seed", str(int(p.get("seed", 1247))),
                    "--video_path", v, "--audio_path", a, "--video_out_path", out],
                   cwd=LATENTSYNC_DIR, check=True)
    return out

def _mimic(image_url, video_url, p):
    img = _dl(image_url, ".jpg"); vid = _dl(video_url, ".mp4")
    od = Path(MIMICMOTION_DIR) / "outputs"; od.mkdir(exist_ok=True)
    cfg = WORK / f"{uuid.uuid4().hex}.yaml"
    cfg.write_text(
        "base_model_path: stabilityai/stable-video-diffusion-img2vid-xt-1-1\n"
        "ckpt_path: models/MimicMotion_1-1.pth\n"
        "test_case:\n"
        f"  - ref_video_path: {vid}\n    ref_image_path: {img}\n"
        f"    num_frames: {int(p.get('frames', 72))}\n    resolution: 576\n    frames_overlap: 6\n"
        f"    num_inference_steps: {int(p.get('steps', 25))}\n    noise_aug_strength: 0\n"
        f"    guidance_scale: {float(p.get('cfg', 2.0))}\n    sample_stride: 2\n"
        f"    fps: {int(p.get('fps', 15))}\n    seed: {int(p.get('seed', 42))}\n")
    before = set(od.glob("*.mp4"))
    subprocess.run(["python", "inference.py", "--inference_config", str(cfg)], cwd=MIMICMOTION_DIR, check=True)
    new = sorted(set(od.glob("*.mp4")) - before, key=lambda x: x.stat().st_mtime) or \
          sorted(od.glob("*.mp4"), key=lambda x: x.stat().st_mtime)
    return str(new[-1])

# Optional stills: advertise + serve `image` ONLY when an image model is truly
# resident. Stills reuse the earlier hybrid-pipeline cells (get_txt2img /
# generate_image); if those cells weren't run, or this session has no usable GPU,
# we don't offer `image` and the worker stays lipsync + motion only.
IMAGE_OK = False
try:
    if "get_txt2img" in globals() and "generate_image" in globals() and HW.get("has_gpu"):
        get_txt2img()  # eager load so we only advertise a model that is actually loaded
        IMAGE_OK = True
        print("[image] stills enabled — model loaded:", _PIPES.get("txt2img_model"))
except Exception as e:
    IMAGE_OK = False
    print(f"[image] stills disabled (no image model loaded): {e}")

CAPS = ["lipsync", "motion", "caption_burn"] + (["image"] if IMAGE_OK else [])

def _image(prompt, imgs, p):
    from PIL import Image
    init = None
    if imgs:
        init = Image.open(_dl(imgs[0], ".png")).convert("RGB")
    img = generate_image(prompt, init_image=init,
                         strength=float(p.get("strength", 0.6)),
                         seed=(int(p["seed"]) if "seed" in p else None),
                         width=int(p.get("width", 512)), height=int(p.get("height", 512)))
    out = str(WORK / f"{uuid.uuid4().hex}.png")
    img.save(out)
    return out

def _caption_burn(video_url, segments, p):
    """Burn timed caption segments into a video using FFmpeg drawtext."""
    import shlex
    vid = _dl(video_url, ".mp4")
    out = str(WORK / f"{uuid.uuid4().hex}.mp4")
    ffmpeg = _ffmpeg_exe()
    # Build a drawtext filter chain — one filter per segment.
    # Each filter is active only between its start/end timestamp.
    filters = []
    for seg in segments:
        t = seg.get("text", "").replace("'", "\'").replace(":", "\:")
        if not t:
            continue
        start = float(seg.get("start", 0))
        end   = float(seg.get("end", start + 3))
        filters.append(
            f"drawtext=text='{t}'"
            f":fontcolor=white:fontsize=24:font=sans-serif"
            f":borderw=2:bordercolor=black@0.8"
            f":x=(w-text_w)/2:y=h-80"
            f":enable='between(t,{start},{end})'"
        )
    if not filters:
        import shutil; shutil.copy(vid, out)
        return out
    vf = ",".join(filters)
    subprocess.run(
        [ffmpeg, "-y", "-i", vid, "-vf", vf,
         "-c:v", "libx264", "-preset", "fast", "-crf", "23",
         "-c:a", "copy", out],
        check=True
    )
    return out


def process_job(job):
    k = (job.get("kind") or "").lower(); p = job.get("params") or {}; imgs = job.get("image_urls") or []
    if k == "lipsync":
        out = _latentsync(job.get("video_url") or (imgs[0] if imgs else None), job.get("audio_url"), p)
    elif k == "motion":
        out = _mimic(imgs[0] if imgs else None, job.get("video_url"), p)
    elif k == "image" and IMAGE_OK:
        out = _image(job.get("prompt") or p.get("prompt") or "", imgs, p)
    elif k == "caption_burn":
        out = _caption_burn(job.get("video_url"), job.get("segments") or [], p)
    else:
        raise ValueError(f"unsupported kind {k!r}; this worker serves {', '.join(CAPS)}")
    return {"url": _up(out)}

# 2) Serve /generate behind a STABLE ngrok tunnel + auto-register in Aurora.
!pip -q install fastapi "uvicorn[standard]" pyngrok nest_asyncio
import threading, time, socket, nest_asyncio, uvicorn
from fastapi import FastAPI, Request, HTTPException
from pyngrok import ngrok
nest_asyncio.apply()
app = FastAPI(title="Aurora GPU worker")
TOKEN = os.environ.get("AURORA_WORKER_TOKEN")

@app.get("/health")
def health(): return {"ok": True, "tasks": CAPS}

@app.post("/generate")
async def generate(req: Request):
    if TOKEN and req.headers.get("authorization") != f"Bearer {TOKEN}":
        raise HTTPException(status_code=401, detail="unauthorized")
    try:
        return process_job(await req.json())
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Start the server first so /health is live before we register.
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()

# Stable tunnel: pin the free static domain so the URL never changes on restart.
if os.environ.get("NGROK_AUTHTOKEN"):
    ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
DOMAIN = os.environ.get("NGROK_STATIC_DOMAIN", "").replace("https://", "").replace("http://", "").rstrip("/")
if DOMAIN:
    ngrok.connect(addr="8000", domain=DOMAIN)  # pin it; let errors surface (no silent fallback)
    public_url = f"https://{DOMAIN}"
else:
    public_url = ngrok.connect(8000).public_url
    print("[ngrok] Set NGROK_STATIC_DOMAIN for a URL that survives restarts.")

# Auto-register in Aurora (apikey = private AURORA_REGISTER_SECRET, NOT the Supabase anon key) once /health answers.
def _register():
    for _ in range(60):
        try:
            requests.get("http://127.0.0.1:8000/health", timeout=2); break
        except Exception:
            time.sleep(1)
    aurora_url = os.environ.get("AURORA_URL", "").rstrip("/")
    key = os.environ.get("AURORA_REGISTER_SECRET", "")
    if not (DOMAIN and aurora_url and key):
        print("[register] skipped — set NGROK_STATIC_DOMAIN, AURORA_URL, AURORA_REGISTER_SECRET."); return
    payload = {"name": os.environ.get("AURORA_WORKER_NAME") or f"colab-{socket.gethostname()}",
               "endpoint_url": f"{public_url}/generate", "protocol": "custom",
               "capabilities": CAPS}
    if TOKEN: payload["auth_token"] = TOKEN
    try:
        r = requests.post(f"{aurora_url}/api/public/workers/register", json=payload,
                          headers={"apikey": key, "content-type": "application/json"}, timeout=30)
        print("[register] OK" if r.ok else f"[register] failed {r.status_code}: {r.text[:200]}")
    except Exception as e:
        print(f"[register] error (worker still serving): {e}")
threading.Thread(target=_register, daemon=True).start()

print("=" * 60)
print(f"Worker live: {public_url}/generate  (caps: {','.join(CAPS)})")
print("Auto-registered in Aurora when AURORA_URL + AURORA_REGISTER_SECRET are set —")
print("otherwise add the printed endpoint manually in Admin -> Workers.")
print("=" * 60)


## 12 · Notes, limits & extensions

**Fallback logic (the "smart engine")**
- `COMPUTE_MODE="auto"` tries the local GPU first. On CUDA OOM / "usage limit"
  errors it (1) clears memory, (2) drops to **low-VRAM + smaller model**, then
  (3) switches to the **external GPU API** if configured.
- Set `COMPUTE_MODE="remote"` to always use the API, or `"local"` to stay on Colab.

**Free-tier tips**
- `sd-turbo` renders a 512² image in ~1-2 steps — ideal for T4 / low VRAM.
- Keep `ENABLE_SVD=False`; Stable Video Diffusion needs ~10 GB and often OOMs.
  The built-in Ken-Burns animator is GPU-free and never crashes.
- Model weights are cached to Drive (`HF_HOME`), so the next session starts fast.

**Reconnects**
- The keep-alive cell stops *idle* disconnects. If Google recycles the runtime,
  just **Run all** again — outputs and the model cache are restored from Drive.

**Real lip-sync (optional, heavier)**
- The audio-driven stage is lightweight amplitude motion, not phoneme lip-sync.
  To upgrade, plug **Wav2Lip** or **SadTalker** into `audio_driven_video()` —
  both need extra model downloads and more VRAM.

**RunPod endpoint contract**
- The client POSTs `{"input": {"prompt": ...}}` to
  `https://api.runpod.ai/v2/<ENDPOINT_ID>/run` and polls `/status/<id>`.
- Your serverless handler should return an image **URL** or **base64** anywhere in
  its `output` (keys like `image`, `image_url`, `images`, `base64` are auto-detected).

**Aurora worker contract**
- Section 13 serves Aurora's flat `/generate` job: `{kind, video_url, audio_url,
  image_urls, params}` in, `{"url": ...}` out (see `workers/CONTRACT.md`).
- `lipsync` runs **LatentSync**, `motion` runs **MimicMotion** — no hosted
  fallback, so register the worker in **Admin → Workers** before using them.
- With a free Ngrok **static domain** + the `AURORA_*` secrets (Section 13 setup),
  the worker keeps the same URL and **re-registers itself** on every restart —
  no Admin -> Workers edit needed.
